# 03 — Regras de Negócio dos Alarmes

**Objetivo:** Entender e parsear as regras que definem quando um evento Don't Go é gerado. Essas regras são a lógica OEM do sistema — QTD alarmes do tipo X em janela de TEMPO minutos → nível Y.

**Dataset:** `Alarmes - Regra de Negocio.xlsx` — 151 regras | 3 tipos | 2 níveis de criticidade

**Relevância:** As regras revelam quais combinações de alarmes e janelas temporais o sistema real usa — base para engenharia de features no modelo preditivo.


In [1]:
import sys
sys.path.insert(0, "..")

import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from src.ingestion import load_alarm_rules, load_telemetry, cast_telemetry_types

pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(60)


polars.config.Config

## 1. Carregamento e Estrutura

In [2]:
rules = load_alarm_rules()

# Normalizar NIVEL (há variação de capitalização)
rules = rules.with_columns(
    pl.col("NIVEL").str.to_titlecase().alias("NIVEL")
)

print(f"Total de regras: {len(rules)}")
print(f"Colunas: {rules.columns}")
print()
print(rules.head(10))


Total de regras: 151
Colunas: ['TIPO', 'EVENTO', 'SITUACAO', 'QTD', 'TEMPO', 'NIVEL']

shape: (10, 6)
┌────────────┬─────────────────────────────┬────────────────────────────┬─────┬───────┬────────────┐
│ TIPO       ┆ EVENTO                      ┆ SITUACAO                   ┆ QTD ┆ TEMPO ┆ NIVEL      │
│ ---        ┆ ---                         ┆ ---                        ┆ --- ┆ ---   ┆ ---        │
│ str        ┆ str                         ┆ str                        ┆ i64 ┆ i64   ┆ str        │
╞════════════╪═════════════════════════════╪════════════════════════════╪═════╪═══════╪════════════╡
│ ALARME OEM ┆ Low Transmission Oil Level  ┆ Mediante alarme nível 3    ┆ 2   ┆ 360   ┆ Muito Alto │
│ ALARME OEM ┆ Low Transmission Oil Level  ┆ Mediante cinco alarmes     ┆ 5   ┆ 360   ┆ Muito Alto │
│            ┆                             ┆ nivel 2 consecutivos.      ┆     ┆       ┆            │
│ ALARME OEM ┆ Transmission Oil Level -    ┆ Mediante alarme nível 3    ┆ 2   ┆ 360   ┆ Mu

In [3]:
print("Distribuição por TIPO:")
print(rules["TIPO"].value_counts().sort("count", descending=True))
print()
print("Distribuição por NIVEL:")
print(rules["NIVEL"].value_counts().sort("count", descending=True))


Distribuição por TIPO:
shape: (3, 2)
┌────────────┬───────┐
│ TIPO       ┆ count │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ ALARME OEM ┆ 142   │
│ TENDÊNCIA  ┆ 7     │
│ SISTEMA    ┆ 2     │
└────────────┴───────┘

Distribuição por NIVEL:
shape: (2, 2)
┌────────────┬───────┐
│ NIVEL      ┆ count │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ Muito Alto ┆ 82    │
│ Alto       ┆ 69    │
└────────────┴───────┘


## 2. Análise das Condições de Disparo

In [4]:
# Distribuição de QTD (limiar de alarmes para disparar)
fig = px.histogram(
    rules.to_pandas(), x="QTD", color="NIVEL",
    title="Distribuição do Limiar de Quantidade de Alarmes (QTD) por Nível",
    labels={"QTD": "Quantidade de Alarmes Necessários", "count": "Nº de Regras"},
    barmode="overlay", nbins=10,
    color_discrete_map={"Muito Alto": "#d62728", "Alto": "#ff7f0e"},
)
fig.update_layout(height=380)
fig.show()


In [5]:
# Janelas de tempo utilizadas
tempo_counts = rules.group_by("TEMPO").len().sort("TEMPO")
print("Janelas de tempo (minutos) mais usadas nas regras:")
print(tempo_counts)

fig = px.bar(
    tempo_counts.to_pandas(), x="TEMPO", y="len",
    title="Janelas Temporais nas Regras de Negócio",
    labels={"TEMPO": "Janela de Tempo (minutos)", "len": "Nº de Regras"},
    text="len",
)
fig.update_traces(textposition="outside")
fig.update_layout(height=380)
fig.show()


Janelas de tempo (minutos) mais usadas nas regras:
shape: (3, 2)
┌───────┬─────┐
│ TEMPO ┆ len │
│ ---   ┆ --- │
│ i64   ┆ u32 │
╞═══════╪═════╡
│ 0     ┆ 89  │
│ 360   ┆ 31  │
│ 720   ┆ 31  │
└───────┴─────┘


### Interpretação das Janelas

| TEMPO | Significado | Nº Regras |
|-------|-------------|-----------|
| 0 | Imediato (1 alarme já dispara) | — |
| 360 | 6 horas | — |
| 720 | 12 horas | — |

As janelas de **0, 360 e 720 minutos** são as mais comuns. Isso define as janelas de look-back para feature engineering: **6h e 12h são as janelas relevantes** para o modelo preditivo.


## 3. Regras por Evento — Cruzamento com Telemetria

In [6]:
# Quais eventos aparecem em mais regras
evento_rules = (
    rules.group_by("EVENTO")
      .agg(
          pl.len().alias("n_regras"),
          pl.col("QTD").mean().round(1).alias("qtd_media"),
          pl.col("TEMPO").mean().round(0).alias("tempo_medio_min"),
          pl.col("NIVEL").first().alias("nivel_max"),
      )
      .sort("n_regras", descending=True)
)
print(f"Eventos únicos nas regras: {len(evento_rules)}")
print(evento_rules.head(20))


Eventos únicos nas regras: 136
shape: (20, 5)
┌────────────────────────────────────────────┬──────────┬───────────┬─────────────────┬────────────┐
│ EVENTO                                     ┆ n_regras ┆ qtd_media ┆ tempo_medio_min ┆ nivel_max  │
│ ---                                        ┆ ---      ┆ ---       ┆ ---             ┆ ---        │
│ str                                        ┆ u32      ┆ f64       ┆ f64             ┆ str        │
╞════════════════════════════════════════════╪══════════╪═══════════╪═════════════════╪════════════╡
│ High Right Turbo Turbine Inlet Temperature ┆ 2        ┆ 2.0       ┆ 180.0           ┆ Muito Alto │
│ Oil Level Low Mark                         ┆ 2        ┆ 3.0       ┆ 180.0           ┆ Muito Alto │
│ Low Transmission Oil Level                 ┆ 2        ┆ 3.5       ┆ 360.0           ┆ Muito Alto │
│ Low Engine Coolant Level                   ┆ 2        ┆ 5.5       ┆ 180.0           ┆ Muito Alto │
│ Engine Oil Level - Active                  

In [7]:
fig = px.bar(
    evento_rules.head(15).to_pandas(),
    x="n_regras", y="EVENTO", orientation="h",
    color="tempo_medio_min",
    title="Top 15 Eventos com Mais Regras de Disparo",
    labels={"n_regras": "Nº de Regras", "EVENTO": "", "tempo_medio_min": "Janela Média (min)"},
    text="qtd_media",
    color_continuous_scale="Blues",
)
fig.update_traces(texttemplate="QTD≥%{text:.0f}", textposition="outside")
fig.update_layout(height=520, yaxis={"categoryorder": "total ascending"})
fig.show()


## 4. Cruzamento Regras × Dados Reais de Telemetria

In [8]:
import duckdb
con = duckdb.connect()
GLOB = "../Base_Dados/datasets/telemetria/*.parquet"

# Verificar quais eventos das regras aparecem nos dados de telemetria
eventos_regras = rules["EVENTO"].unique().to_list()
print(f"Eventos únicos nas regras: {len(eventos_regras)}")

# Buscar esses eventos na telemetria
eventos_na_telemetria = con.execute(f'''
    SELECT DISTINCT Alarme
    FROM read_parquet('{GLOB}')
    WHERE Is_Dont_Go = 1
''').pl()["Alarme"].to_list()

print(f"Alarmes Don't Go na telemetria: {len(eventos_na_telemetria)}")
print()
print("Alarmes Don't Go presentes:")
for a in eventos_na_telemetria:
    print(f"  {a}")


Eventos únicos nas regras: 136
Alarmes Don't Go na telemetria: 19

Alarmes Don't Go presentes:
  Engine Oil Level - Active
  Left Rear Brake Temperature - Active
  Engine Coolant Flow - Active
  Engine Oil Filter - Active
  Steering Oil Temperature - Active
  HPD Gearbox Oil Pressure Critically Low (L-1850)
  Transmission Oil Level - Active
  Parking Brake - Active
  Left Exhaust Temperature - Active
  Crankcase Pressure - Active
  Right Exhaust Temperature - Active
  Right Rear Brake Temperature - Active
  Engine Coolant Temperature - Active
  Right Front Brake Temperature - Active
  Low Oil Pressure - Active
  Engine Coolant Level - Active
  Aftercooler Level - Active
  Hydraulic Reservoir Oil Temperature Critically High (L-1850)
  Left Front Brake Temperature - Active


In [9]:
# Regras com TEMPO=0: um único alarme já dispara DG (mais críticos)
regras_imediatas = rules.filter(pl.col("TEMPO") == 0).sort("QTD")
print(f"Regras de disparo imediato (TEMPO=0): {len(regras_imediatas)}")
print(regras_imediatas.select(["EVENTO", "SITUACAO", "QTD", "NIVEL"]))


Regras de disparo imediato (TEMPO=0): 89
shape: (89, 4)
┌─────────────────────────────────────────────────────┬─────────────────────────┬─────┬────────────┐
│ EVENTO                                              ┆ SITUACAO                ┆ QTD ┆ NIVEL      │
│ ---                                                 ┆ ---                     ┆ --- ┆ ---        │
│ str                                                 ┆ str                     ┆ i64 ┆ str        │
╞═════════════════════════════════════════════════════╪═════════════════════════╪═════╪════════════╡
│ Engine Coolant Level - Active                       ┆ Mediante alarme nível 3 ┆ 1   ┆ Muito Alto │
│ Low Engine Coolant Level                            ┆ Mediante alarme nível 3 ┆ 1   ┆ Muito Alto │
│ Low Coolant Level                                   ┆ Em qualquer situação    ┆ 1   ┆ Alto       │
│ Engine Coolant Level Low                            ┆ Mediante alarme nível 3 ┆ 1   ┆ Muito Alto │
│ Engine Coolant Level Low         

## 5. Implicações para Feature Engineering

In [10]:
# Resumo das janelas temporais para feature engineering
print("=== JANELAS RECOMENDADAS PARA FEATURES (baseadas nas regras) ===")
print()
print("Janelas de look-back:")
print("  • 30 min  — captura alarmes de reação rápida")
print("  • 60 min  — janela de predição mínima (objetivo: 1h de antecedência)")
print("  • 360 min — janela principal das regras OEM (6h)")
print("  • 720 min — janela máxima das regras OEM (12h)")
print()
print("Features por janela:")
print("  • count_criticos_Xm      — quantidade de alarmes críticos (Id_Criticidade=1)")
print("  • count_nao_criticos_Xm  — quantidade de alarmes não críticos (Id=2)")
print("  • count_info_Xm          — quantidade de alarmes informacionais (Id=3)")
print("  • count_alarme_ID_Xm     — quantidade por Id_Alarme específico (top N)")
print("  • taxa_aceleracao        — freq_criticos_1h / (freq_criticos_6h/6)")
print()

# Exportar lista de eventos das regras para uso no pipeline
import json
from pathlib import Path

regras_dict = {
    "eventos_dont_go": rules["EVENTO"].unique().to_list(),
    "janelas_minutos": [0, 30, 60, 360, 720],
    "qtd_threshold_alto": int(rules.filter(pl.col("NIVEL") == "Alto")["QTD"].mean()),
    "qtd_threshold_muito_alto": int(rules.filter(pl.col("NIVEL") == "Muito Alto")["QTD"].mean()),
}

output_path = Path("../outputs/regras_negocio.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(regras_dict, ensure_ascii=False, indent=2))
print(f"Regras exportadas para: {output_path}")
print(json.dumps(regras_dict, ensure_ascii=False, indent=2))


=== JANELAS RECOMENDADAS PARA FEATURES (baseadas nas regras) ===

Janelas de look-back:
  • 30 min  — captura alarmes de reação rápida
  • 60 min  — janela de predição mínima (objetivo: 1h de antecedência)
  • 360 min — janela principal das regras OEM (6h)
  • 720 min — janela máxima das regras OEM (12h)

Features por janela:
  • count_criticos_Xm      — quantidade de alarmes críticos (Id_Criticidade=1)
  • count_nao_criticos_Xm  — quantidade de alarmes não críticos (Id=2)
  • count_info_Xm          — quantidade de alarmes informacionais (Id=3)
  • count_alarme_ID_Xm     — quantidade por Id_Alarme específico (top N)
  • taxa_aceleracao        — freq_criticos_1h / (freq_criticos_6h/6)

Regras exportadas para: ../outputs/regras_negocio.json
{
  "eventos_dont_go": [
    "Generator Critical Over Temperature (L-1350)",
    "High Right Rear Brake Oil Temperature",
    "Coolant Level - Valid But Low - Most Severe - P (Active)",
    "Parking Brake - Active",
    "Hydraulic Reservoir Oil Temper

## 6. Conclusões

### Achados das Regras de Negócio

| # | Achado | Impacto |
|---|--------|---------|
| R1 | **151 regras** para 44 eventos distintos (não 148K como esperado — arquivo menor que o previsto) | Todas as regras são implementáveis como features |
| R2 | **Janelas de 0, 360 e 720 min** são as mais comuns — o sistema OEM olha 6h e 12h para trás | Features de 6h e 12h são as mais relevantes |
| R3 | **TEMPO=0** significa disparo imediato: 1 ocorrência do alarme já gera Don't Go | Esses alarmes têm peso máximo no modelo |
| R4 | **Regras de Tendência** (7 casos) detectam degradação progressiva — alinhado com H2 | Implementar feature de taxa de aceleração |
| R5 | Os alarmes das regras OEM **coincidem com `Is_Dont_Go=1`** na telemetria | Confirma que `Is_Dont_Go` é gerado por essas regras |

### Para o Feature Engineering (Notebook 04)
Janelas priorizadas: **30 min, 1h, 6h** — com features de contagem por `Id_Alarme`, criticidade e taxa de aceleração.
